# crop-mesh-detector on Colab

Runs on a free NVIDIA GPU instead of your laptop's CPU. PlantVillage is fetched straight from
the `tensorflow_datasets` library on Colab's fast network — no manual data transfer.

**Before running anything**: `Runtime` menu -> `Change runtime type` -> Hardware accelerator -> `T4 GPU` -> Save.

In [ ]:
# Confirm the GPU runtime is actually attached
!nvidia-smi

## 1. Clone the project (private repo)

This repo is private, so cloning it here needs a GitHub personal access token —
generate one at github.com -> Settings -> Developer settings -> Personal access tokens
(fine-grained, read-only access to this one repo is enough). `getpass` keeps it out of
the notebook's saved output/history.

In [ ]:
from getpass import getpass

token = getpass('GitHub personal access token: ')
!git clone https://{token}@github.com/karboon1008/crop-mesh-detector.git /content/crop-mesh-detector
del token
%cd /content/crop-mesh-detector
!ls

## 2. Install dependencies

Colab already ships recent `torch`/`tensorflow`/`numpy`; this adds what's missing
(`timm`, `codecarbon`) and pins the rest per `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

## 3. Fetch PlantVillage

Same script as local — pulls the dataset via the `tensorflow_datasets` library
catalog entry (no Kaggle account needed) and writes it out as `.jpg` files under
`data/PlantVillage/`. Takes a few minutes on Colab's network.

In [ ]:
!python scripts/download_plantvillage.py

## 4. Train

`src/train.py` picks `cuda` automatically when available
(`device = "cuda" if torch.cuda.is_available() else "cpu"`) — on Colab's GPU runtime
that resolves to the T4, no code change needed.

Edit `config.yaml` first (below) for a quicker first pass — e.g. trim `models.architectures`
to one entry, or lower `training.baseline_epochs` / `training.rounds`.

In [ ]:
# Optional: inspect/edit the config before running
!cat config.yaml

In [ ]:
!python -m src.train --config config.yaml

## 5. Download results

Zips everything in `outputs/` (accuracy/collaboration-gain JSON, `emissions.csv`,
the sustainability report) and prompts a browser download back to your Mac.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/outputs', 'zip', 'outputs')
files.download('/content/outputs.zip')